# Step 3: char embedding
`CharEmbedder` is tested here (one notebook per stage).
The class itself lives in `08_encoder_step.py`.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from importlib import import_module
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import torch
from torch import nn

from helpers.pairs import load_pairs

_vocab_step = import_module("siamese_model.01_vocab_step")       # digit-prefixed filenames -> can't use `from ... import`
build_vocab = _vocab_step.build_vocab
PAD_ID, UNK_ID = _vocab_step.PAD_ID, _vocab_step.UNK_ID
encode_batch = import_module("siamese_model.03_encode_batch_step").encode_batch


In [2]:
# the class lives in 08_encoder_step.py (single copy, shared with training); this notebook tests it
CharEmbedder = import_module("siamese_model.08_encoder_step").CharEmbedder

## Tests

In [3]:
ot_train, ot_train_label = load_pairs("overture", "train")   # only the train split: the vocab is built from train
vocab = build_vocab(ot_train)

In [4]:
torch.manual_seed(0)
ids, lengths = encode_batch(["abcd", "b", "c", "?", "!"], vocab)
embedder = CharEmbedder(len(vocab))
output = embedder(ids)

assert output.shape == (5, 4, 32)                       # (B, T, emb_dim)
assert torch.equal(output[3], output[4])                # "?" and "!" are both unk -> same vector (a lookup copies rows, so exact equality is fine)
assert torch.equal(output[1, 1:], torch.zeros(3, 32))   # padding positions embed to all-zero vectors

# parameters: checked by name, plus the total count, so nothing extra slips in unchecked
params = dict(embedder.named_parameters())
assert params["emb.weight"].shape == (len(vocab), 32)
assert sum(p.numel() for p in embedder.parameters()) == len(vocab) * 32   # 40 x 32 = 1,280

# gradients: a lookup only sends gradient to the rows that were used
output.sum().backward()
grad = params["emb.weight"].grad
assert torch.equal(grad[vocab["w"]], torch.zeros(32))   # "w" not in the batch -> no gradient
assert grad[vocab["a"]].abs().sum() > 0                 # used character -> nonzero gradient
assert grad[UNK_ID].abs().sum() > 0                     # unk row is used ("?", "!") -> it trains too
assert torch.equal(grad[PAD_ID], torch.zeros(32))       # pad was used, but padding_idx keeps its row frozen
print("char embedding: all passed")

char embedding: all passed
